In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import gradio as gr
from openai import OpenAI

In [ ]:

def greet(name):
  return "Hello " + name + "!"

demo = gr.Interface(fn=greet, inputs="textbox", outputs="textbox", flagging_mode="never")

# demo.launch()

In [ ]:
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

system_message = "You are a helpful assistant"

def message_llama(prompt):
  messages = [{"role": "system", "content": system_message}, {"role": "user", "content": prompt}]
  response = openai.chat.completions.create(model="llama3.2:3b", messages=messages)
  return response.choices[0].message.content

# message_llama("What is today's date?")

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message for Llama 3.2", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
  fn=message_llama,
  title="Llama",
  inputs=[message_input],
  outputs=[message_output],
  examples=["hello", "howdy"],
  flagging_mode="never"
)

# view.launch()

In [ ]:
def stream_llama(prompt):
  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": prompt}
  ]
  stream = openai.chat.completions.create(
    model="llama3.2:3b",
    messages=messages,
    stream=True
  )
  result = ""
  for chunk in stream:
    result += chunk.choices[0].delta.content or ""
    yield result

In [ ]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your message:", info="Enter a message for Llama 3.2", lines=7)
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
  fn=stream_llama,
  title="Llama",
  inputs=[message_input],
  outputs=[message_output],
  examples=[
    "Explain the Transformer architecture to a layperson",
    "Explain the Transformer architecture to an aspiring AI engineer",
  ],
  flagging_mode="never"
)

view.launch()